In [1]:
import sys
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("CWD:", os.getcwd())
print("PROJECT_ROOT:", PROJECT_ROOT)
print("CONFIGS EXISTS:", (PROJECT_ROOT / "configs").exists())
print("SRC EXISTS:", (PROJECT_ROOT / "src").exists())

CWD: /home/harielpadillasanchez/Documentos/TT/TT2/notebooks
PROJECT_ROOT: /home/harielpadillasanchez/Documentos/TT/TT2
CONFIGS EXISTS: True
SRC EXISTS: True


In [2]:
from configs.models import MODELS, DEFAULT_GENERATION_CONFIG
from configs.prompts import ZERO_SHOT_TEMPLATE, build_few_shot_prompt
from src.experiment.schemas import new_record
from src.utils.logging_utils import ExperimentLogger
from src.experiment.runner import ExperimentRunner


/home/harielpadillasanchez/Documentos/TT/TT2/.venv-bloom/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
experiment_id = "exp_inference_v1"

manifest = {
    "experiment_id": experiment_id,
    "goal": "Prueba base de pipeline para simplificación textual con Ollama y Transformers",
    "models": MODELS,
    "default_generation_config": DEFAULT_GENERATION_CONFIG,
    "notes": [
        "Primera prueba funcional del pipeline",
        "Mistral y Llama3 por Ollama",
        "BLOOMZ por Transformers",
        "Aún sin métricas automáticas"
    ]
}

logger = ExperimentLogger(base_dir="outputs/logs")
logger.save_manifest(experiment_id, manifest)

print("Manifest guardado correctamente.")

Manifest guardado correctamente.


In [4]:
source_text = "Una vez dirimidos estos asuntos, se entra de lleno a algunos aspectos e instrumentos que son propios de las finanzas."

prompt_r2 = ZERO_SHOT_TEMPLATE.format(
    source=source_text,
    rules_block="Guía interna de simplificación: usa palabras comunes; evita tecnicismos innecesarios; reemplaza palabras difíciles por palabras simples; usa oraciones cortas; expresa una idea por oración; prefiere voz activa; divide oraciones largas.\n"
)

print(prompt_r2)

Reescribe en español el siguiente texto con lenguaje más claro y sencillo.
Conserva el significado original y no inventes información.
Guía interna de simplificación: usa palabras comunes; evita tecnicismos innecesarios; reemplaza palabras difíciles por palabras simples; usa oraciones cortas; expresa una idea por oración; prefiere voz activa; divide oraciones largas.

Devuelve solo la versión final simplificada.

Texto:
Una vez dirimidos estos asuntos, se entra de lleno a algunos aspectos e instrumentos que son propios de las finanzas.

Versión simplificada:


In [5]:
record = new_record(
    experiment_id=experiment_id,
    dataset_name="FEINA",
    model_key="mistral",
    model_id=MODELS["mistral"]["model_id"],
    backend="ollama",
    prompt_type="zero-shot",
    ruleset="R2",
    generation_config=DEFAULT_GENERATION_CONFIG,
    source_text=source_text,
    reference_text=None,
    sample_id="demo_001",
    prompt_text=prompt_r2,
)

logger.append_record(experiment_id, record.to_dict())

print("Record de prueba guardado.")
print(record.to_dict()["run_id"])

Record de prueba guardado.
92e7bd0a-f585-444a-a2a7-4f8156995300


In [6]:
runner = ExperimentRunner(experiment_id=experiment_id)
print("Runner listo.")

Runner listo.


In [7]:
source_text = "Una vez dirimidos estos asuntos, se entra de lleno a algunos aspectos e instrumentos que son propios de las finanzas."

record_mistral = runner.run_one(
    dataset_name="FEINA",
    model_key="mistral",
    prompt_type="zero-shot",
    ruleset="R2",
    source_text=source_text,
    sample_id="demo_mistral_001",
    generation_config=DEFAULT_GENERATION_CONFIG,
)

print("STATUS:", record_mistral.status)
print("TIME:", record_mistral.inference_seconds)
print("OUTPUT:")
print(record_mistral.generated_text)

STATUS: success
TIME: 5.7255
OUTPUT:
Despues de solucionar estos temas, entramos en detalle sobre algunos aspectos y herramientas especificas de las finanzas.


In [8]:
print("PROMPT USADO:")
print(record_mistral.prompt_text)
print("\n" + "="*80 + "\n")
print("OUTPUT LIMPIO:")
print(record_mistral.generated_text)
print("\n" + "="*80 + "\n")
print("ERROR:")
print(record_mistral.error_message)

PROMPT USADO:
Reescribe en español el siguiente texto con lenguaje más claro y sencillo.
Conserva el significado original y no inventes información.
Guía interna de simplificación: usa palabras comunes; evita tecnicismos innecesarios; reemplaza palabras difíciles por palabras simples; usa oraciones cortas; expresa una idea por oración; prefiere voz activa; divide oraciones largas.

Devuelve solo la versión final simplificada.

Texto:
Una vez dirimidos estos asuntos, se entra de lleno a algunos aspectos e instrumentos que son propios de las finanzas.

Versión simplificada:


OUTPUT LIMPIO:
Despues de solucionar estos temas, entramos en detalle sobre algunos aspectos y herramientas especificas de las finanzas.


ERROR:
None


In [9]:
record_llama3 = runner.run_one(
    dataset_name="FEINA",
    model_key="llama3",
    prompt_type="zero-shot",
    ruleset="R2",
    source_text=source_text,
    sample_id="demo_llama3_001",
    generation_config=DEFAULT_GENERATION_CONFIG,
)

print("STATUS:", record_llama3.status)
print("TIME:", record_llama3.inference_seconds)
print("OUTPUT:")
print(record_llama3.generated_text)

STATUS: success
TIME: 8.8053
OUTPUT:
Una vez resueltos estos problemas, podemos hablar sobre los temas y herramientas financieros.


In [10]:
print("STATUS:", record_llama3.status)
print("ERROR MESSAGE:")
print(record_llama3.error_message)

STATUS: success
ERROR MESSAGE:
None


In [11]:
print("PROMPT USADO:")
print(record_mistral.prompt_text)
print("\n" + "="*80 + "\n")
print("OUTPUT LIMPIO:")
print(record_mistral.generated_text)
print("\n" + "="*80 + "\n")
print("ERROR:")
print(record_mistral.error_message)

PROMPT USADO:
Reescribe en español el siguiente texto con lenguaje más claro y sencillo.
Conserva el significado original y no inventes información.
Guía interna de simplificación: usa palabras comunes; evita tecnicismos innecesarios; reemplaza palabras difíciles por palabras simples; usa oraciones cortas; expresa una idea por oración; prefiere voz activa; divide oraciones largas.

Devuelve solo la versión final simplificada.

Texto:
Una vez dirimidos estos asuntos, se entra de lleno a algunos aspectos e instrumentos que son propios de las finanzas.

Versión simplificada:


OUTPUT LIMPIO:
Despues de solucionar estos temas, entramos en detalle sobre algunos aspectos y herramientas especificas de las finanzas.


ERROR:
None


In [12]:
record_bloomz = runner.run_one(
    dataset_name="FEINA",
    model_key="bloomz_small",
    prompt_type="zero-shot",
    ruleset="R2",
    source_text=source_text,
    sample_id="demo_bloomz_001",
    generation_config=DEFAULT_GENERATION_CONFIG,
)

print("STATUS:", record_bloomz.status)
print("TIME:", record_bloomz.inference_seconds)
print("OUTPUT:")
print(record_bloomz.generated_text)

STATUS: success
TIME: 3.1471
OUTPUT:
Ahora puedes empezar tu propia empresa.


In [13]:
from pathlib import Path

log_path = Path("outputs/logs") / f"{experiment_id}.jsonl"
print("Existe log:", log_path.exists())
print("Ruta:", log_path)

if log_path.exists():
    with open(log_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    print("Número de registros:", len(lines))
    print("Último registro:")
    print(lines[-1][:1500])

Existe log: True
Ruta: outputs/logs/exp_inference_v1.jsonl
Número de registros: 46
Último registro:
{"experiment_id": "exp_inference_v1", "run_id": "59e11949-fa10-46bb-be79-a44593714927", "timestamp": "2026-03-08T06:26:12.785065", "dataset_name": "FEINA", "fold_id": null, "split_name": null, "model_key": "bloomz_small", "model_id": "bigscience/bloomz-560m", "backend": "transformers", "prompt_type": "zero-shot", "ruleset": "R2", "few_shot_example_ids": [], "generation_config": {"temperature": 0.2, "top_p": 0.9, "max_new_tokens": 256, "repetition_penalty": 1.05, "do_sample": false}, "sample_id": "demo_bloomz_001", "source_text": "Una vez dirimidos estos asuntos, se entra de lleno a algunos aspectos e instrumentos que son propios de las finanzas.", "reference_text": null, "generated_text": "Ahora puedes empezar tu propia empresa.", "prompt_text": "Reescribe en español el siguiente texto con lenguaje más claro y sencillo.\nConserva el significado original y no inventes información.\nGuía i

In [14]:
runner.full_cleanup()
print("Limpieza completa.")

Limpieza completa.


In [15]:
print("=== MISTRAL ===")
print(record_mistral.generated_text)
print()
print("=== LLAMA3 ===")
print(record_llama3.generated_text)
print("=== BLOOMZ ===")
print(record_bloomz.generated_text)

=== MISTRAL ===
Despues de solucionar estos temas, entramos en detalle sobre algunos aspectos y herramientas especificas de las finanzas.

=== LLAMA3 ===
Una vez resueltos estos problemas, podemos hablar sobre los temas y herramientas financieros.
=== BLOOMZ ===
Ahora puedes empezar tu propia empresa.


In [16]:
print(record_bloomz.error_message)

None
